# Session 39 — Hands-On (Simple Edition)
## Logging, Monitoring & Analytics for LLMs

**What we'll build:** a tiny chat function that calls **Groq**, then watch it two ways:

1. **Prometheus + Grafana** → numbers over time (latency, tokens, cost)
2. **Langfuse** → full prompt + response for each call, with a UI to replay

Both are **free** and **open-source**. By the end you'll have wired both into the same LLM call.

---
## 0. Setup

Install the libraries we need. (Run this once.)

In [1]:
!pip install -q groq prometheus_client langfuse python-dotenv requests

### 0.1 API keys

Put your keys in a `.env` file in the same folder as this notebook:

```
GROQ_API_KEY=gsk_...
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
```

- **Groq** → free key at <https://console.groq.com/>
- **Langfuse** → free key at <https://cloud.langfuse.com/> (sign up → project settings)

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env into os.environ

print('Groq key set:    ', bool(os.environ.get('GROQ_API_KEY')))
print('Langfuse pub set:', bool(os.environ.get('LANGFUSE_PUBLIC_KEY')))
print('Langfuse sec set:', bool(os.environ.get('LANGFUSE_SECRET_KEY')))

Groq key set:     True
Langfuse pub set: True
Langfuse sec set: True


### 0.2 Quick smoke test — one Groq call

Confirm the Groq client works before we add anything fancy.

In [3]:
from groq import Groq

groq_client = Groq()  # reads GROQ_API_KEY from env
MODEL = 'llama-3.1-8b-instant'  # fast + cheap

resp = groq_client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Say hi in exactly 5 words.'}],
)

print('Reply:', resp.choices[0].message.content)
print('Tokens used:', resp.usage.total_tokens)

Reply: Hello from the computer machine.
Tokens used: 50


---
# Part 1 — Prometheus + Grafana

From the slides: **Prometheus = the scoreboard**, **Grafana = the chart maker**.

We'll teach our LLM call to write down 4 numbers every time it runs:

| Metric | Type | Why |
|---|---|---|
| `llm_requests_total` | Counter | How many calls? |
| `llm_tokens_total` | Counter | How many tokens (= cost driver)? |
| `llm_cost_usd_total` | Counter | Running $ cost |
| `llm_latency_seconds` | Histogram | How slow is each call? |

### 1.1 Define the metrics

Each metric has a name and (optionally) **labels** so we can slice by model, status, etc.

In [4]:
from prometheus_client import Counter, Histogram, REGISTRY

# Notebook re-run safety: unregister any existing llm_* metrics first.
# Without this, re-running the cell raises 'Duplicated timeseries'.
for c in {c for n, c in REGISTRY._names_to_collectors.items() if n.startswith('llm_')}:
    REGISTRY.unregister(c)

requests_total = Counter(
    'llm_requests_total', 'Total LLM requests',
    labelnames=['model', 'status'],
)

tokens_total = Counter(
    'llm_tokens_total', 'Total tokens used',
    labelnames=['model', 'kind'],  # kind = input or output
)

cost_usd_total = Counter(
    'llm_cost_usd_total', 'Total cost in USD',
    labelnames=['model'],
)

latency_seconds = Histogram(
    'llm_latency_seconds', 'How long each LLM call took',
    labelnames=['model'],
)

print('Metrics ready ✅')

Metrics ready ✅


### 1.2 Wrap the LLM call

After every call we update the metrics. That's it — no Prometheus server needed yet, the library keeps the numbers in memory.

In [5]:
import time

# Groq pricing per 1M tokens (check console.groq.com for current prices)
PRICE_PER_1M = {
    'llama-3.1-8b-instant': {'input': 0.05, 'output': 0.08},
}

def ask_with_metrics(prompt: str, model: str = MODEL) -> str:
    start = time.time()
    resp = groq_client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
    )

    # Record metrics
    latency_seconds.labels(model=model).observe(time.time() - start)
    requests_total.labels(model=model, status='ok').inc()
    tokens_total.labels(model=model, kind='input').inc(resp.usage.prompt_tokens)
    tokens_total.labels(model=model, kind='output').inc(resp.usage.completion_tokens)

    p = PRICE_PER_1M[model]
    cost = (resp.usage.prompt_tokens * p['input']
            + resp.usage.completion_tokens * p['output']) / 1_000_000
    cost_usd_total.labels(model=model).inc(cost)

    return resp.choices[0].message.content

### 1.3 Generate some traffic

Make a few calls so the metrics have something to show.

In [26]:
prompts = [
    "What's the capital of Indonesia?",
    "Explain machine learning in one sentence.",
    "Translate 'good morning' to Japanese.",
    "What is 17 * 23?",
    "Give me one productivity tip.",
]

for p in prompts:
    answer = ask_with_metrics(p)
    print(f'Q: {p}')
    print(f'A: {answer[:80]}...\n')

Q: What's the capital of Indonesia?
A: The capital of Indonesia is Jakarta....

Q: Explain machine learning in one sentence.
A: Machine learning is a type of artificial intelligence that enables systems to le...

Q: Translate 'good morning' to Japanese.
A: 'good morning' in Japanese is 'ohayou gozaimasu'. 

Here's a breakdown of how it...

Q: What is 17 * 23?
A: 17 * 23 = 391....

Q: Give me one productivity tip.
A: Here's a simple yet effective productivity tip:

**The 2-Minute Rule**: If a tas...



### 1.4 Read the metrics — same format Prometheus will see

`generate_latest()` returns the exact text Prometheus would scrape from your app.

In [7]:
from prometheus_client import generate_latest

print(generate_latest().decode())

# HELP python_gc_objects_collected_total Objects collected during gc
# TYPE python_gc_objects_collected_total counter
python_gc_objects_collected_total{generation="0"} 1846.0
python_gc_objects_collected_total{generation="1"} 537.0
python_gc_objects_collected_total{generation="2"} 165.0
# HELP python_gc_objects_uncollectable_total Uncollectable objects found during GC
# TYPE python_gc_objects_uncollectable_total counter
python_gc_objects_uncollectable_total{generation="0"} 0.0
python_gc_objects_uncollectable_total{generation="1"} 0.0
python_gc_objects_uncollectable_total{generation="2"} 0.0
# HELP python_gc_collections_total Number of times this generation was collected
# TYPE python_gc_collections_total counter
python_gc_collections_total{generation="0"} 279.0
python_gc_collections_total{generation="1"} 25.0
python_gc_collections_total{generation="2"} 2.0
# HELP python_info Python platform information
# TYPE python_info gauge
python_info{implementation="CPython",major="3",minor="11",pa

**Notice:**

- `llm_requests_total{model="llama-3.1-8b-instant",status="ok"} 5.0` — we made 5 successful calls
- `llm_tokens_total{...kind="input"}` and `kind="output"` — split by direction
- `llm_latency_seconds_bucket{...}` — Prometheus stores latency in buckets for fast percentile queries
- `llm_cost_usd_total{...}` — a tiny number, but it accumulates fast in production

### 1.5 Expose `/metrics` for Prometheus to scrape

In production, your app exposes an HTTP endpoint. Prometheus visits it every ~15 seconds and saves the numbers. One line starts that server:

In [8]:
from prometheus_client import start_http_server
import requests as http

start_http_server(8000)  # /metrics is now live at localhost:8000

# Let's call it ourselves to see what Prometheus would see:
metrics_text = http.get('http://localhost:8000/metrics').text
# Show only our custom LLM metrics (skip Python's default process metrics)
for line in metrics_text.splitlines():
    if line.startswith('llm_') or line.startswith('# HELP llm_'):
        print(line)

# HELP llm_requests_total Total LLM requests
llm_requests_total{model="llama-3.1-8b-instant",status="ok"} 5.0
# HELP llm_requests_created Total LLM requests
llm_requests_created{model="llama-3.1-8b-instant",status="ok"} 1.779903518830841e+09
# HELP llm_tokens_total Total tokens used
llm_tokens_total{kind="input",model="llama-3.1-8b-instant"} 212.0
llm_tokens_total{kind="output",model="llama-3.1-8b-instant"} 326.0
# HELP llm_tokens_created Total tokens used
llm_tokens_created{kind="input",model="llama-3.1-8b-instant"} 1.779903518830859e+09
llm_tokens_created{kind="output",model="llama-3.1-8b-instant"} 1.7799035188308768e+09
# HELP llm_cost_usd_total Total cost in USD
llm_cost_usd_total{model="llama-3.1-8b-instant"} 3.668e-05
# HELP llm_cost_usd_created Total cost in USD
llm_cost_usd_created{model="llama-3.1-8b-instant"} 1.77990351883089e+09
# HELP llm_latency_seconds How long each LLM call took
llm_latency_seconds_bucket{le="0.005",model="llama-3.1-8b-instant"} 0.0
llm_latency_seconds_b

### 1.6 Plug Grafana in

We've pre-wired Prometheus + Grafana for you in the `monitoring/` folder next to this notebook.
Open a terminal and run:

```bash
cd monitoring
docker compose up -d
```

Then open:

| Service    | URL                     | Login         |
|------------|-------------------------|---------------|
| Prometheus | http://localhost:9090   | (none)        |
| Grafana    | http://localhost:3000   | admin / admin |

**To verify scraping works:** open <http://localhost:9090/targets> — you should see `llm-app` with state **UP** (as long as section 1.5 above is still running and exposing port 8000).

**To build your first chart:** Grafana → Dashboards → New → Add visualization → Prometheus (already configured) → paste a query below → Run.

| Question | PromQL |
|---|---|
| Requests per minute | `rate(llm_requests_total[1m]) * 60` |
| Cost burned today | `increase(llm_cost_usd_total[24h])` |
| p95 latency | `histogram_quantile(0.95, rate(llm_latency_seconds_bucket[5m]))` |
| Error rate | `rate(llm_requests_total{status="error"}[5m]) / rate(llm_requests_total[5m])` |

When you're done: `docker compose down` (add `-v` to also wipe Grafana's data).

### 1.7 Tutorial — Build a RED dashboard in Grafana

**RED** is the classic observability pattern for any service:

- **R**ate — requests per second
- **E**rrors — fraction that fails
- **D**uration — how slow

These three numbers tell you 80% of what you need to know. Let's build a 4-panel RED dashboard for our LLM app step by step.

> **Prerequisite:** `docker compose up -d` is running in `monitoring/` and section 1.5 above is still active so `/metrics` is being scraped.

---

#### Step 1 — Open Grafana

1. Open <http://localhost:3000>
2. Login: `admin` / `admin` → skip password change
3. Left sidebar → **Dashboards** → click **New → New dashboard**
4. Click **+ Add visualization** → select **Prometheus** (already provisioned)

---

#### Step 2 — Panel 1: Rate (requests / min)

- **Visualization** (top-right dropdown): `Stat`
- **Query** (Code editor):
  ```promql
  sum(rate(llm_requests_total[5m])) * 60
  ```
- **Title** (right pane → Panel options): `Requests / min`
- **Unit** (right pane → Standard options → Unit): `reqps`
- Click **Apply** (top-right)

---

#### Step 3 — Panel 2: Errors (error rate %)

- **Add** → **Visualization** → `Stat`
- **Query:**
  ```promql
  sum(rate(llm_requests_total{status="error"}[5m]))
  /
  sum(rate(llm_requests_total[5m]))
  ```
- **Title:** `Error rate`
- **Unit:** `Percent (0.0–1.0)` — so `0.04` displays as `4%`
- **Thresholds** (right pane → Thresholds):
  - green: `Base`
  - yellow: `0.01`
  - red: `0.05`
- **Apply**

---

#### Step 4 — Panel 3: Duration (p95 latency)

- **Add** → **Visualization** → `Stat`
- **Query:**
  ```promql
  histogram_quantile(0.95, sum(rate(llm_latency_seconds_bucket[5m])) by (le))
  ```
- **Title:** `p95 latency`
- **Unit:** `seconds (s)`
- **Apply**

---

#### Step 5 — Panel 4: Latency over time (line chart)

- **Add** → **Visualization** → `Time series`
- Three queries (click `+ Query` to add B and C):

  | Ref | PromQL | Legend |
  |---|---|---|
  | A | `histogram_quantile(0.50, sum(rate(llm_latency_seconds_bucket[5m])) by (le))` | `p50` |
  | B | `histogram_quantile(0.95, sum(rate(llm_latency_seconds_bucket[5m])) by (le))` | `p95` |
  | C | `histogram_quantile(0.99, sum(rate(llm_latency_seconds_bucket[5m])) by (le))` | `p99` |

- **Title:** `Latency over time`
- **Unit:** `seconds (s)`
- **Apply**

---

#### Step 6 — Save the dashboard

- Top-right → click the **Save** icon (💾)
- Name: `LLM RED dashboard`
- Save

---

#### Step 7 — Generate traffic and watch it move

Re-run the loop in **section 1.3** (the `for p in prompts` cell) a few times.
The dashboard auto-refreshes every 15 seconds — or click the refresh icon top-right.

**What "good" looks like:**
- **Rate** — stable, no surprise spikes
- **Errors** — green, ideally 0
- **p95** — steady, not creeping up over time
- **Latency lines** — `p50` and `p95` close together = consistent UX. `p99` ≫ `p95` = some users having a bad day

---

#### Going further — beyond RED

For LLMs, three more panels are worth adding to the same dashboard:

| Panel | PromQL | Visualization |
|---|---|---|
| Cost / minute | `sum(rate(llm_cost_usd_total[5m])) * 60` | Stat or Time series |
| Tokens / sec by kind | `sum by (kind) (rate(llm_tokens_total[5m]))` | Time series |
| Cost by model | `sum by (model) (rate(llm_cost_usd_total[5m])) * 60` | Pie chart |

**After Part 4** you'll also have `pipeline_*` metrics. Add a second dashboard or extra row with:
- Rejection rate: `sum(rate(pipeline_requests_total{status="rejected"}[5m])) / sum(rate(pipeline_requests_total{step="screening"}[5m]))`
- p95 latency by step: `histogram_quantile(0.95, sum by (le, step) (rate(pipeline_latency_seconds_bucket[5m])))`

---
# Part 2 — Langfuse

Prometheus tells you **"how fast"** and **"how many"**. Langfuse tells you **"what exactly did the model see and say"** — per call.

Think of Langfuse as the **diary for your AI** with a beautiful UI on top.

### 2.1 Initialize the client

Reads `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY` from env automatically.

In [9]:
from langfuse import Langfuse

langfuse = Langfuse()  # picks up keys from env
print('Auth OK:', langfuse.auth_check())

Auth OK: True


### 2.2 Wrap the LLM call

Use `start_as_current_observation` as a `with` block. Everything inside the block is one **generation** in Langfuse — input, output, tokens, cost, all linked.

In [10]:
def ask_with_langfuse(prompt: str, model: str = MODEL) -> str:
    with langfuse.start_as_current_observation(
        name='groq-chat',
        as_type='generation',
        model=model,
        input=prompt,
    ) as gen:
        resp = groq_client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
        )
        answer = resp.choices[0].message.content

        gen.update(
            output=answer,
            usage_details={
                'input': resp.usage.prompt_tokens,
                'output': resp.usage.completion_tokens,
                'total': resp.usage.total_tokens,
            },
        )
        return answer

### 2.3 Run a few calls

Each call becomes one trace in the Langfuse UI. We `flush()` at the end so data is sent before the notebook moves on.

In [11]:
for p in [
    'Name 3 Indonesian foods.',
    'What is the largest ocean?',
    'Suggest a Python project for a beginner.',
]:
    print('→', ask_with_langfuse(p)[:80], '...')

langfuse.flush()
print('\nOpen https://cloud.langfuse.com → Traces to see them')

→ Here are 3 popular Indonesian foods:

1. **Nasi Goreng**: Indonesian-style fried ...
→ The largest of the world's five oceans is the Pacific Ocean. It covers about 155 ...
→ **Project: To-Do List App**

Here's a fun and engaging project for a beginner in ...

Open https://cloud.langfuse.com → Traces to see them


### 2.4 What to look for in the Langfuse UI

1. **Traces** tab → click any row → see the full prompt + response
2. **Costs** column → Langfuse computes it from `usage_details` + model price
3. **Latency** column → automatically measured by the `with` block
4. **Scores** → add eval scores to traces (good/bad, helpful/not) — useful later for A/B testing (Session 40)

### 2.5 Beyond traces — sessions, users, and prompts

Tracing is just step one. Three more features make Langfuse useful in real ops:

| Feature | What | Why it matters |
|---|---|---|
| **`user_id`** | Tag every trace with the user | Filter the UI by user — debug *Bob's* bad experience, not all 10,000 calls |
| **`session_id`** | Group traces into a session | A multi-turn chat is **one** session — see the whole conversation in one screen |
| **Prompts** | Store prompts IN Langfuse, fetch them in code | Edit prompts in the UI without redeploying. Every trace links to the exact prompt version that produced it |

**Prompts are the killer feature.** When a user complains *"your bot got weird yesterday"*, you can see exactly which prompt version they hit and roll back. This is why teams move prompts out of code.

### 2.6 Store the two pipeline prompts in Langfuse

We'll create the **screening** prompt (Part 4's first LLM) and the **answer** prompt (Part 4's RAG LLM) as managed prompts. Each gets a name, type (`chat`), and a `{{variable}}` placeholder for runtime data.

The `ensure_prompt` helper is **idempotent** — first run creates the prompt, every subsequent run just fetches it. So this cell is safe to re-run.

In [12]:
from langfuse import propagate_attributes

def ensure_prompt(name, prompt, type='chat', labels=('production',)):
    """Fetch the prompt if it exists in Langfuse; otherwise create it."""
    try:
        return langfuse.get_prompt(name, type=type)
    except Exception:
        print(f'Prompt "{name}" not found — creating v1 now...')
        return langfuse.create_prompt(
            name=name, prompt=prompt, type=type, labels=list(labels),
        )

# --- Prompt for the SCREENING LLM (first LLM in Part 4) ---
SCREEN_PROMPT = ensure_prompt(
    name='helpdesk-screening',
    type='chat',
    prompt=[
        {'role': 'system', 'content': (
            'You are a ticket screener for an IT helpdesk. '
            'Reply ONLY with a JSON object — no explanation.\n'
            'Schema:\n'
            '{\n'
            '  "language":       "id" | "en" | "other",\n'
            '  "category":       "access" | "network" | "hardware" | "email" | "sap" | "software" | "other",\n'
            '  "tone":           "normal" | "urgent" | "abusive",\n'
            '  "is_it_question": true | false\n'
            '}'
        )},
        {'role': 'user', 'content': '{{query}}'},
    ],
)

# --- Prompt for the ANSWER LLM (second LLM in Part 4) ---
ANSWER_PROMPT = ensure_prompt(
    name='helpdesk-answer',
    type='chat',
    prompt=[
        {'role': 'user', 'content': (
            'You are a helpful IT helpdesk assistant for Mitsubishi BSI. '
            'Use ONLY the SOPs below. Cite the SOP ID(s) in your answer.\n\n'
            'SOPs:\n{{context}}\n\n'
            'User question: {{query}}'
        )},
    ],
)

print(f'Screening prompt → v{SCREEN_PROMPT.version}')
print(f'Answer prompt    → v{ANSWER_PROMPT.version}')
print('Both prompts now live in Langfuse → Prompts tab.')

Prompt 'helpdesk-screening-label:production' not found during refresh, evicting from cache.


Prompt "helpdesk-screening" not found — creating v1 now...


Prompt 'helpdesk-answer-label:production' not found during refresh, evicting from cache.


Prompt "helpdesk-answer" not found — creating v1 now...
Screening prompt → v1
Answer prompt    → v1
Both prompts now live in Langfuse → Prompts tab.


### 2.7 Mini demo — fetch, compile, and use a stored prompt

`prompt.compile(query='...')` returns the rendered list of messages, ready to pass straight to Groq.

We also use `propagate_attributes` — a context manager that sets `user_id`, `session_id`, and `tags` **once** at the top of a request, and Langfuse auto-applies them to every nested observation underneath.

In [13]:
def ask_with_session(query: str, user_id: str, session_id: str) -> dict:
    with langfuse.start_as_current_observation(
        name='session-chat', as_type='span', input=query,
    ) as root:
        with propagate_attributes(
            user_id=user_id,
            session_id=session_id,
            tags=['demo', 'screening-only'],
        ):
            messages = SCREEN_PROMPT.compile(query=query)  # ← variables substituted

            with langfuse.start_as_current_observation(
                name='screening', as_type='generation',
                model=MODEL, input=messages,
            ) as gen:
                r = groq_client.chat.completions.create(
                    model=MODEL, messages=messages,
                    response_format={'type': 'json_object'},
                )
                verdict = json.loads(r.choices[0].message.content)
                gen.update(
                    output=verdict,
                    prompt=SCREEN_PROMPT,  # ← links this trace to prompt vN in the UI
                    usage_details={
                        'input':  r.usage.prompt_tokens,
                        'output': r.usage.completion_tokens,
                        'total':  r.usage.total_tokens,
                    },
                )
            root.update(output=verdict)
            return verdict

import json

# Simulate a 3-turn 'session' with one user
session = 'sess-demo-001'
for q in [
    'My VPN is broken',
    'Actually never mind, my password expired',
    'Can you reset it for me?',
]:
    print(q, '→', ask_with_session(q, user_id='wira', session_id=session))

langfuse.flush()
print(f'\nOpen Langfuse → Sessions → look for "{session}"')
print('All 3 traces will be grouped under one session.')

My VPN is broken → {'language': 'en', 'category': 'network', 'tone': 'normal', 'is_it_question': False}
Actually never mind, my password expired → {'language': 'en', 'category': 'access', 'tone': 'normal', 'is_it_question': False}
Can you reset it for me? → {'language': 'id', 'category': 'access', 'tone': 'normal', 'is_it_question': True}

Open Langfuse → Sessions → look for "sess-demo-001"
All 3 traces will be grouped under one session.


---
# Part 3 — Putting Both Together

Real production code does **both** in one function: Prometheus for ops dashboards, Langfuse for AI debugging.

In [14]:
def ask_full(prompt: str, user_id: str = 'anon', model: str = MODEL) -> str:
    start = time.time()

    with langfuse.start_as_current_observation(
        name='groq-chat',
        as_type='generation',
        model=model,
        input=prompt,
    ) as gen:
        resp = groq_client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
        )
        answer = resp.choices[0].message.content
        latency = time.time() - start

        # → Prometheus (numbers)
        latency_seconds.labels(model=model).observe(latency)
        requests_total.labels(model=model, status='ok').inc()
        tokens_total.labels(model=model, kind='input').inc(resp.usage.prompt_tokens)
        tokens_total.labels(model=model, kind='output').inc(resp.usage.completion_tokens)

        # → Langfuse (full event with cost + user)
        gen.update(
            output=answer,
            user_id=user_id,
            usage_details={
                'input': resp.usage.prompt_tokens,
                'output': resp.usage.completion_tokens,
                'total': resp.usage.total_tokens,
            },
        )
        return answer

# Demo
for prompt, uid in [
    ('Tell a 1-line joke about programmers.', 'wira'),
    ("Translate 'thank you' to Indonesian.",  'budi'),
]:
    print(f'[{uid}] →', ask_full(prompt, user_id=uid))

langfuse.flush()

[wira] → Why do programmers prefer dark mode? Because light attracts bugs.
[budi] → The translation of "thank you" to Indonesian is:

- Terima kasih (official formal phrase)
- Trima kaseh (colloquial and informal)
- Baiklah (very casual and can be used with close friends or family)


---
# Part 4 — A More Realistic Pipeline (Screening + RAG)

So far we've watched **one** LLM call. Real systems chain LLMs together.
Let's build a small IT helpdesk pipeline using the SOP docs from Session 37:

```
[user message]
      ↓
[Screening LLM — small/fast]   ← language, tone, category, is-IT?
      ├── reject  →  return early (save $$)
      └── pass    →  continue
                ↓
         [Chroma — find top 2 SOPs]
                ↓
         [Answer LLM — bigger model + the SOPs]
                ↓
         [sourced answer]
```

**Why two models?** Screening is high-volume and simple → use a cheap fast model. Answering needs reasoning → use a bigger model. **You only pay 10× more for queries that earn it.**

**Why track every step?** When something is slow or expensive, you want to know *which* step. Langfuse will show one nested trace per request; Prometheus will give us per-step latency and cost.

### 4.1 Install Chroma

Chroma is a free open-source vector database. The default embedding model (`all-MiniLM-L6-v2`) is downloaded automatically on first use — no API key needed.

In [15]:
!pip install -q chromadb

### 4.2 Load the SOP docs

Pull every `.md` file from `Session 37/datasets/sop-docs/` and parse the `**Category:**` line into metadata so we can filter later.

In [16]:
from pathlib import Path

SOP_DIR = Path('../Session 37/datasets/sop-docs')
sop_files = sorted(SOP_DIR.glob('*.md'))

docs, metas, ids = [], [], []
for f in sop_files:
    text = f.read_text()
    category = 'general'
    for line in text.splitlines()[:10]:
        if line.startswith('**Category:**'):
            category = line.split(':**', 1)[1].strip().lower()
            break
    docs.append(text)
    metas.append({'file': f.name, 'category': category})
    ids.append(f.stem)

print(f'Loaded {len(docs)} SOPs:')
for m in metas:
    print(f"  {m['file']:<45}  category={m['category']}")

Loaded 13 SOPs:
  SOP-001-VPN-FortiClient.md                     category=network
  SOP-002-MFA-Token-Reset.md                     category=access
  SOP-003-Password-Reset.md                      category=access
  SOP-004-SAP-Login-Issues.md                    category=erp
  SOP-005-SAP-Transaction-Errors.md              category=erp
  SOP-006-Outlook-Email-Issues.md                category=other
  SOP-007-Network-Printer-Setup.md               category=network
  SOP-008-Laptop-Hardware-Request.md             category=hardware
  SOP-009-Software-Install-Request.md            category=other
  SOP-010-New-Employee-Onboarding.md             category=other
  SOP-011-Account-Termination.md                 category=access
  SOP-012-Teams-M365-Access.md                   category=access
  SOP-013-Corporate-WiFi.md                      category=network


### 4.3 Chunking — split each doc into smaller pieces

If we embed a 5-page SOP as **one** vector, that vector is a smudge — it represents "a bit of everything in the doc." When a user asks a specific question, the search may return the wrong doc because the matching paragraph got averaged out.

**The fix:** split each doc into smaller chunks and embed each chunk separately. Retrieval then matches at chunk level — much sharper.

**Rules of thumb for LLM RAG:**
- Chunk size: ~200–500 tokens (≈ 1–3 paragraphs)
- Overlap chunks by ~10–20% so a sentence at the boundary isn't lost
- Keep section headers IN the chunk so the LLM has context

Our IT SOPs already have natural section headers (`## Purpose`, `## Resolution Steps`, ...). Splitting on `## ` gives us clean chunks for free — no need for fancy token-based splitting today.

In [17]:
import re

def chunk_md(text: str, doc_id: str, file_name: str, category: str) -> list[dict]:
    """Split a markdown doc by '## ' headers. Each chunk keeps the doc title for context."""
    # Doc title is the first '# ...' line
    title_match = re.search(r'^# (.+)', text, flags=re.M)
    title = title_match.group(1).strip() if title_match else doc_id

    # Split into parts at each '## ' (preserving the header)
    parts = re.split(r'\n(?=## )', text)

    chunks = []
    for i, part in enumerate(parts):
        part = part.strip()
        if not part:
            continue
        # Prepend the doc title to every chunk except the one that already has it
        if not part.startswith('# '):
            part = f'[{title}]\n\n{part}'
        chunks.append({
            'id':   f'{doc_id}__{i}',
            'text': part,
            'meta': {'doc_id': doc_id, 'file': file_name,
                     'category': category, 'chunk': i},
        })
    return chunks

all_chunks = []
for doc, meta, doc_id in zip(docs, metas, ids):
    all_chunks.extend(chunk_md(doc, doc_id, meta['file'], meta['category']))

print(f'{len(docs)} docs  →  {len(all_chunks)} chunks')
print(f'Avg chunk size: {sum(len(c["text"]) for c in all_chunks) // len(all_chunks)} chars')
print()
print('Example chunk:')
print('-' * 60)
print(all_chunks[2]['text'][:400] + '...')

13 docs  →  112 chunks
Avg chunk size: 439 chars

Example chunk:
------------------------------------------------------------
[SOP-001: VPN Access via FortiClient]

## Scope
Applies to all BSI-managed laptops running FortiClient v7.0 or later. Excludes BYOD devices (route to SOP-014 Mobile Enrollment)....


### 4.4 Embeddings — what model does Chroma use?

An **embedding** turns a piece of text into a list of numbers (a vector). Texts that mean similar things get vectors that point in similar directions. That's how vector search works.

Chroma's default embedding model is **`all-MiniLM-L6-v2`** from Sentence-Transformers:

| Property | Value |
|---|---|
| Type | Local — no API, no cost |
| Vector dimension | 384 |
| Size on disk | ~90 MB (downloaded once) |
| Speed | ~1000 sentences/sec on CPU |
| Quality | Good English; decent Indonesian (BERT was trained on both) |

**When to swap:**
- Need stronger multilingual → `paraphrase-multilingual-MiniLM-L12-v2`
- Need OpenAI-quality (paid) → `text-embedding-3-small` (1536-dim)
- Need higher quality, English-heavy → `all-mpnet-base-v2` (768-dim)

Let's confirm what's being used and how to swap it.

In [18]:
from chromadb.utils import embedding_functions

default_ef = embedding_functions.DefaultEmbeddingFunction()
sample = default_ef(['hello world'])
print(f'Embedding class:    {default_ef.__class__.__name__}')
print(f'Vector dimension:   {len(sample[0])}')
print(f'First 8 values:     {sample[0][:8]}')

# --- Swap to a multilingual model (optional) ---
# multi_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
#     model_name='paraphrase-multilingual-MiniLM-L12-v2',
# )
# Then pass it when creating the collection:
#   chroma.create_collection(name='sop_docs', embedding_function=multi_ef)

Embedding class:    DefaultEmbeddingFunction
Vector dimension:   384
First 8 values:     [-0.03447726  0.03102333  0.00673505  0.02610899 -0.03936202 -0.16030248
  0.06692406 -0.00644149]


### 4.5 Index the chunks in Chroma

One call. Chroma embeds and stores every chunk for us.

In [19]:
import chromadb

chroma = chromadb.Client()  # in-memory; resets when the kernel restarts
collection = chroma.create_collection(name='sop_docs')

collection.add(
    documents=[c['text'] for c in all_chunks],
    metadatas=[c['meta'] for c in all_chunks],
    ids=[c['id']         for c in all_chunks],
)
print(f'Indexed {collection.count()} chunks.')

Indexed 112 chunks.


### 4.6 Quick retrieval sanity check

Before we add LLMs, confirm the vector search returns the right SOP **section**. Notice that `id` is now `doc_id__chunk_index` — finer-grained than whole docs.

In [20]:
results = collection.query(
    query_texts=['my password expired and I cannot log in'],
    n_results=3,
)
for cid, dist, meta in zip(
    results['ids'][0], results['distances'][0], results['metadatas'][0]
):
    print(f'  {cid:<40}  category={meta["category"]:<10}  dist={dist:.3f}')

  SOP-003-Password-Reset__3                 category=access      dist=0.996
  SOP-003-Password-Reset__1                 category=access      dist=1.007
  SOP-003-Password-Reset__0                 category=access      dist=1.015


### 4.7 Pipeline-specific Prometheus metrics

We want to slice latency and cost by **pipeline step** (`screening`, `retrieval`, `answer`). Easiest: a new family of metrics with a `step` label. The single-call metrics from Part 1 still keep working.

In [21]:
# Same re-run safety as before — clear any existing pipeline_* metrics first.
for c in {c for n, c in REGISTRY._names_to_collectors.items() if n.startswith('pipeline_')}:
    REGISTRY.unregister(c)

pipeline_requests = Counter(
    'pipeline_requests_total', 'Pipeline step results',
    labelnames=['step', 'status'],
)
pipeline_tokens = Counter(
    'pipeline_tokens_total', 'Pipeline tokens by step',
    labelnames=['step', 'kind'],
)
pipeline_latency = Histogram(
    'pipeline_latency_seconds', 'Pipeline latency by step',
    labelnames=['step'],
)
print('Pipeline metrics ready ✅')

Pipeline metrics ready ✅


### 4.8 The full pipeline

Reads top to bottom: screening → gate → retrieval → answer. Every step is wrapped in `langfuse.start_as_current_observation`, which **auto-nests** because we're inside an outer `with` block. After each step we also bump Prometheus.

In [22]:
import json

SCREEN_MODEL = 'llama-3.1-8b-instant'      # fast + cheap
ANSWER_MODEL = 'llama-3.3-70b-versatile'   # better reasoning

# Note: prompts now come from Langfuse (created in section 2.6).
# To edit them, change the prompt in the Langfuse UI — no redeploy.

def handle_ticket(query: str, user_id: str = 'anon',
                  session_id: str | None = None) -> dict:
    session_id = session_id or f'sess-adhoc-{int(time.time())}'

    with langfuse.start_as_current_observation(
        name='helpdesk-ticket', as_type='span', input=query,
    ) as root:
        with propagate_attributes(
            user_id=user_id,
            session_id=session_id,
            tags=['helpdesk', 'pipeline-v1'],
        ):
            # ---------- 1) SCREENING ----------
            t = time.time()
            screen_msgs = SCREEN_PROMPT.compile(query=query)
            with langfuse.start_as_current_observation(
                name='screening', as_type='generation',
                model=SCREEN_MODEL, input=screen_msgs,
            ) as obs:
                r = groq_client.chat.completions.create(
                    model=SCREEN_MODEL,
                    messages=screen_msgs,
                    response_format={'type': 'json_object'},
                )
                verdict = json.loads(r.choices[0].message.content)
                obs.update(
                    output=verdict,
                    prompt=SCREEN_PROMPT,  # ← link to prompt version
                    usage_details={
                        'input':  r.usage.prompt_tokens,
                        'output': r.usage.completion_tokens,
                        'total':  r.usage.total_tokens,
                    },
                )
            pipeline_latency.labels(step='screening').observe(time.time() - t)
            pipeline_tokens.labels(step='screening', kind='input').inc(r.usage.prompt_tokens)
            pipeline_tokens.labels(step='screening', kind='output').inc(r.usage.completion_tokens)
            pipeline_requests.labels(step='screening', status='ok').inc()

            # ---------- 2) GATE ----------
            if not verdict.get('is_it_question') or verdict.get('tone') == 'abusive':
                pipeline_requests.labels(step='pipeline', status='rejected').inc()
                out = {'status': 'rejected', 'reason': 'non-IT or abusive',
                       'screening': verdict}
                root.update(output=out, level='WARNING')
                return out

            # ---------- 3) RETRIEVAL ----------
            t = time.time()
            with langfuse.start_as_current_observation(
                name='retrieval', as_type='retriever', input=query,
            ) as obs:
                hits = collection.query(query_texts=[query], n_results=3)
                chunks  = hits['documents'][0]
                sources = hits['ids'][0]
                obs.update(output={'sources': sources, 'n_chunks': len(chunks)})
            pipeline_latency.labels(step='retrieval').observe(time.time() - t)

            # ---------- 4) ANSWER ----------
            t = time.time()
            context = '\n\n---\n\n'.join(chunks)
            answer_msgs = ANSWER_PROMPT.compile(context=context, query=query)
            with langfuse.start_as_current_observation(
                name='answer', as_type='generation',
                model=ANSWER_MODEL, input=answer_msgs,
            ) as obs:
                r = groq_client.chat.completions.create(
                    model=ANSWER_MODEL,
                    messages=answer_msgs,
                )
                answer = r.choices[0].message.content
                obs.update(
                    output=answer,
                    prompt=ANSWER_PROMPT,  # ← link to prompt version
                    usage_details={
                        'input':  r.usage.prompt_tokens,
                        'output': r.usage.completion_tokens,
                        'total':  r.usage.total_tokens,
                    },
                )
            pipeline_latency.labels(step='answer').observe(time.time() - t)
            pipeline_tokens.labels(step='answer', kind='input').inc(r.usage.prompt_tokens)
            pipeline_tokens.labels(step='answer', kind='output').inc(r.usage.completion_tokens)
            pipeline_requests.labels(step='answer', status='ok').inc()
            pipeline_requests.labels(step='pipeline', status='ok').inc()

            out = {'status': 'ok', 'screening': verdict,
                   'sources': sources, 'answer': answer}
            root.update(output=out)
            return out

### 4.9 Try it — with sessions and users

Two users, multi-turn sessions. Each `(user_id, session_id)` tuple groups its traces in the Langfuse UI.

**Watch the cost difference:** rejected queries finish in one Groq call (just screening). Approved queries do three (screening + retrieval + answer).

In [23]:
# (user_id, session_id, query) — same session = multi-turn conversation
demos = [
    ('wira', 'sess-wira-001', 'Saya lupa password Active Directory, bagaimana reset-nya?'),
    ('wira', 'sess-wira-001', 'Apakah bisa via telepon atau harus pakai Teams?'),  # follow-up
    ('budi', 'sess-budi-001', 'VPN FortiClient will not connect from home — help!'),
    ('budi', 'sess-budi-001', 'How do I install Visual Studio Code on a corporate laptop?'),
    ('troll','sess-troll-001','You IT people are useless. Fix my laptop NOW.'),  # abusive → rejected
    ('troll','sess-troll-001','What is the best pizza place nearby?'),            # off-topic → rejected
]

for user_id, session_id, q in demos:
    print(f'\n{"=" * 70}')
    print(f'USER: {user_id:<6}  SESSION: {session_id}')
    print(f'QUERY: {q}')
    out = handle_ticket(q, user_id=user_id, session_id=session_id)
    print(f'STATUS:    {out["status"]}')
    print(f'SCREENING: {out["screening"]}')
    if out['status'] == 'ok':
        print(f'SOURCES:   {out["sources"]}')
        print(f'ANSWER:    {out["answer"][:250]}...')
    else:
        print(f'REASON:    {out["reason"]}')

langfuse.flush()

print('\n' + '=' * 70)
print('Open Langfuse:')
print('  • Sessions → wira/budi/troll each have their own session view')
print('  • Users    → filter by user_id; see all of wira\'s traces in one place')
print('  • Prompts  → helpdesk-screening, helpdesk-answer — both with version + linked traces')


USER: wira    SESSION: sess-wira-001
QUERY: Saya lupa password Active Directory, bagaimana reset-nya?
STATUS:    ok
SCREENING: {'language': 'id', 'category': 'access', 'tone': 'normal', 'is_it_question': True}
SOURCES:   ['SOP-003-Password-Reset__0', 'SOP-003-Password-Reset__3', 'SOP-003-Password-Reset__1']
ANSWER:    Untuk mereset password Active Directory, Anda perlu mengikuti prosedur yang tertuang dalam SOP-003. Silakan hubungi tim BSI Identity & Access Management untuk melakukan reset password. Pastikan Anda menyediakan informasi yang diperlukan untuk memveri...

USER: wira    SESSION: sess-wira-001
QUERY: Apakah bisa via telepon atau harus pakai Teams?
STATUS:    ok
SCREENING: {'language': 'id', 'category': 'other', 'tone': 'normal', 'is_it_question': True}
SOURCES:   ['SOP-013-Corporate-WiFi__2', 'SOP-013-Corporate-WiFi__1', 'SOP-007-Network-Printer-Setup__2']
ANSWER:    Maaf, saya tidak dapat menemukan informasi tentang metode kontak di SOP yang disediakan. SOP yang disediakan

### 4.10 What you should see in your dashboards

**In Langfuse** (https://cloud.langfuse.com → Traces):
- Each query is ONE trace named `helpdesk-ticket` with nested children: `screening`, optionally `retrieval` + `answer`
- Rejected queries have **only** the `screening` child — cheap, fast
- Approved queries show the full tree, with cost split across screening + answer

**In Grafana** (http://localhost:3000 — make sure `docker compose up` is running):

| Question | PromQL |
|---|---|
| Tokens by pipeline step | `sum by (step) (rate(pipeline_tokens_total[5m]))` |
| Rejection rate | `rate(pipeline_requests_total{status="rejected"}[5m]) / rate(pipeline_requests_total{step="screening"}[5m])` |
| p95 latency per step | `histogram_quantile(0.95, sum by (le, step) (rate(pipeline_latency_seconds_bucket[5m])))` |
| Cost saved by screening | `rate(pipeline_requests_total{status="rejected"}[5m])` (each rejection ≈ saves one big-model call) |

**This is the payoff** of all the instrumentation work: when traffic spikes or cost jumps, you can answer *which step* in *which language* for *which category* — in seconds.

---
## Mini Challenge

Pick one or more and try it:

1. **Track errors.** Wrap the Groq call in `try/except` — on failure, `requests_total.labels(status='error').inc()` and `gen.update(level='ERROR')`.
2. **Add a `use_case` label** to the Prometheus metrics (e.g., `chat` vs `summarize`) so the Grafana dashboard can slice cost by feature.
3. **Tag traces with a session.** Pass `session_id` to `gen.update(...)` — then in Langfuse, group multi-turn conversations together.
4. **Plot it.** Call `ask_full(...)` 50 times in a loop with random prompts, then read `generate_latest()` and parse `llm_cost_usd_total` to plot cost growth with matplotlib.

### Key takeaways

- ✅ **One LLM call ⇒ two systems**: Prometheus + Langfuse, in one function.
- ✅ **Prometheus is for ops** — dashboards, alerts, SLAs.
- ✅ **Langfuse is for AI debugging** — replay any conversation, score it, search by user.
- ✅ **Cost lives in both** — Prometheus shows totals over time; Langfuse shows per-call detail.
- ⚠️ Never log raw user data without a redaction step (see slide 25).